<a href="https://colab.research.google.com/github/TomerRippin/Machine-Learning/blob/master/DeepLearning-Basics-simple-pytorch-implementations.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

Maman 11 - Deep  Learning

In [ ]:
import torch
import numpy as np

from itertools import zip_longest

Assigment 1: expand_as function

In [ ]:
def expand_as(A, B):
  """
  function that expands the tensor A to the shape of tensor B.
  :returns: C - expanded tensor
  """
  # If in the same shape, return a copy of A as C
  if A.shape == B.shape:
    return A.clone()

  # Cannot expand bigger vector to a smaller shape
  elif len(A.shape) > len(B.shape):
    raise ValueError("Shapes are not compatible! the shape of A must be \
     smaller or equal to B")

  else:
    # Check for the broadcasting conditions
    for a, b in zip(A.shape[::-1], B.shape[::-1]):
      if a == b or a == 1:
        pass
      else:
        raise RuntimeError(f"Shapes are not compatible! Mismatching dimensions dim A: {a}, dim B:{b}")

  # Create a Copy of A
  C = A.clone()

  # Expand C to the number of dimensions as B
  for i in range(len(B.shape) - len(A.shape)):
      C = torch.unsqueeze(C, 0)

  # Multiply C by the dimension size of B in locations where dim C == 1 and Dim B != 1.
  for i in range(len(B.shape)):
    if C.shape[i] == 1 and B.shape[i] > 1:
      C = torch.cat([C] * B.shape[i], dim=i)
  return C


Assigment 2 + 3: broadcast_tensors

In [ ]:
def are_tensors_broadcastable(A, B):
  """
  function that checks if two tensors are broadcastable.
  :returns: (True, shape) if broadcastable and to what shape, (False, None) otherwise
  """
  if A.shape == B.shape:
    return True, A.shape

  # Follow Broadcast Semantics and create the shape
  shape = []
  for a, b in zip_longest(A.shape[::-1], B.shape[::-1]):
    # Broadcast semantics check
    if a == b:
      shape.append(a)
    elif a == 1:
      shape.append(b)
    elif b == 1:
      shape.append(a)

    # One of the shapes is longer, complete the final shape with the remaining data
    elif a is None:
      shape.append(b)
    elif b is None:
      shape.append(a)

    # Did not follow any of the broadcasting rules
    else:
      return False, None

  # The check is complete, the tensors are broadcastable
  return True, shape[::-1]


In [ ]:
def broadcast_pair(A, B):
  """
  function that broadcasts two tensors to the bigger of them.
  :returns: C, D - pair of broadcasted tensors
  """
  if A.shape == B.shape:
    return A.clone(), B.clone()

  # Check if broadcastable and what size is the output.
  broadcastable, shape = are_tensors_broadcastable(A, B)
  if not broadcastable:
    raise RuntimeError("Tensors are not broadcastable!")

  # Create a Dummy tensor with the desierd final shape
  dummy = torch.ones(shape)

  # Expand the tensors to match the shape of the dummy tensor
  C = expand_as(A, dummy)
  D = expand_as(B, dummy)

  return C, D

Assigment 4: Tests

In [ ]:
!pip install ipytest

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.6/1.6 MB 20.8 MB/s eta 0:00:00


In [ ]:
import pytest
import ipytest

ipytest.autoconfig()

# ----------------------------------------------

def test_expand_as_logic():
    """Tests basic expansion and singleton dimension expansion."""
    # Case 1: Standard expansion
    a = torch.tensor([[1], [2], [3]]) # (3, 1)
    b = torch.empty(3, 4)             # (3, 4)

    expected = a.expand_as(b)
    actual = expand_as(a, b)

    assert actual.shape == expected.shape
    assert torch.equal(actual, expected)

    # Case 2: Expanding a scalar-like tensor
    a_scalar = torch.tensor([5])
    b_large = torch.empty(2, 2, 2)
    assert torch.equal(expand_as(a_scalar, b_large), a_scalar.expand_as(b_large))

def test_expand_as_errors():
    """Tests for illegal expansions."""
    a = torch.randn(2, 2)
    b = torch.randn(2, 3) # Cannot expand 2 to 3

    with pytest.raises(RuntimeError):
        expand_as(a, b)

def test_broadcast_tensors_multi():
    """Should be braodcasted together """
    t1 = torch.randn(1, 2, 4)
    t2 = torch.randn(2, 1, 4)

    expected_1, expected_2 = torch.broadcast_tensors(t1, t2)
    actual_1, actual_2 = broadcast_pair(t1, t2)

    assert actual_1.shape == (2, 2, 4)
    assert actual_2.shape == (2, 2, 4)
    assert torch.equal(actual_1, expected_1)
    assert torch.equal(actual_2, expected_2)

def test_broadcast_incompatible():
    """Tests shapes that simply cannot be broadcasted."""
    t1 = torch.randn(2, 3)
    t2 = torch.randn(2, 4)

    with pytest.raises(RuntimeError):
        broadcast_pair(t1, t2)

def test_edge_case_scalars():
    """Tests broadcasting a tensor with a 0-dim scalar."""
    t1 = torch.randn(5, 5)
    t2 = torch.tensor(1.0)

    act1, act2 = broadcast_pair(t1, t2)
    assert act2.shape == (5, 5)
    assert (act2 == 1.0).all()

ipytest.run()

.....                                                                                        [100%]
5 passed in 0.18s


<ExitCode.OK: 0>